# 홍대 상권 방문 시뮬레이션 (2단계 프롬프트, n=500)

이태원 실험(`캡스톤_논문실험2.ipynb`)과 동일한 구조이되, **LLM 호출을 OpenAI 대신 Google Gemini(무료 티어)로** 바꿨습니다. 그 외 차이는:
- 상권 정보(1단계 JSON)만 홍대용으로 교체
- 실제 유동인구 CSV는 **홍대입구역(홍대), 2026 Q1** 한 행을 업로드
- 숙제 요구사항대로, 2단계(LLM 방문판단) 결과 외에 **"2단계 없이 페르소나 풀에서 그냥 무작위 추출한 원본 분포"를 베이스라인으로 추가** — 10대 제외 정규화가 2단계 유무와 무관하게 얼마나 오차를 줄이는지, 그리고 2단계 자체가 베이스라인보다 얼마나 오차를 줄이는지를 분리해서 봅니다.

> 주의 1: 베이스라인은 **연령대·성별에만** 유효합니다. 시간대·요일은 페르소나 원본 필드에 없고 2단계 LLM 판단으로만 생성되는 값이라, "2단계 없이 시간대/요일 분포"는 애초에 정의되지 않습니다. 이 두 축은 2단계 결과만 실제값과 비교합니다.
>
> 주의 2: **모델이 다릅니다.** 동대문·이태원 예시는 GPT 계열(gpt-5-mini류)을 썼고, 이 노트북은 무료로 돌리기 위해 Gemini(`gemini-3.5-flash-lite`)를 씁니다. 그래서 이 결과를 다른 상권 결과들과 "같은 조건" 비교로 쓰기는 어렵고, 홍대 자체의 경향(성별·연령은 잘 맞고 시간대·요일은 안 맞는지 등)을 보는 용도로 생각하시면 됩니다. 모델을 맞추고 싶어지면 나중에 OpenAI 키로 바꿔서 다시 돌릴 수 있도록 함수 구조는 그대로 유지했습니다.
> 무료 티어는 분당 요청수(RPM)·일일 요청수 한도가 있어서, 아래 코드는 호출마다 일부러 쉬어가며(느리게) 돕니다. 한도 초과 오류가 나면 잠깐 쉬었다가 같은 배치를 재시도하도록 되어 있습니다.

In [ ]:
!pip install -q google-genai datasets pandas numpy tqdm

import json, re, random, time
import numpy as np
import pandas as pd

from datasets import load_dataset
from google import genai
from google.genai import types
from google.colab import userdata, files
from IPython.display import display

# 기본 설정
SEED = 42
TARGET_VISITORS = 150      # 2단계에서 채울 방문 수락자 수
BASELINE_N = TARGET_VISITORS   # 베이스라인(무작위 추출) 표본 크기 — 방문수락자 수와 항상 동일하게 맞춤
BATCH_SIZE = 10
MODEL = "gemini-3.5-flash-lite"   # lite 계열이 flash 계열보다 무료 티어 RPM(분당 요청 수)이 넉넉한 편입니다.
                                   # 그래도 429가 계속 뜨면 https://aistudio.google.com/rate-limit 에서
                                   # 이 프로젝트의 실제 한도를 확인하고, 그 한도에 맞게 MIN_SECONDS_BETWEEN_CALLS를 늘리세요.

# 무료 티어 RPM(분당 요청) 한도를 피하기 위한 최소 호출 간격(초).
# 너무 빨리 돌다가 429(rate limit) 에러가 잦으면 이 값을 늘리세요.
MIN_SECONDS_BETWEEN_CALLS = 15   # 첫 시도라 넉넉하게 잡았습니다. aistudio.google.com/rate-limit에서
                                  # 실제 RPM을 확인한 뒤 60 / RPM 값에 여유를 조금 더한 값으로 바꿔도 됩니다.

random.seed(SEED)
np.random.seed(SEED)

# Google AI Studio에서 발급받은 API 키를
# Colab 왼쪽 열쇠 아이콘(Secrets)에 GEMINI_API_KEY 이름으로 등록해두세요.
client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

_last_call_time = [0.0]

def paced_generate(prompt, model=MODEL, max_retries=6):
    """RPM 한도를 지키면서 generate_content 호출. 429/일시 오류 시 지수 백오프 재시도."""
    for attempt in range(max_retries):
        wait = MIN_SECONDS_BETWEEN_CALLS - (time.time() - _last_call_time[0])
        if wait > 0:
            time.sleep(wait)
        try:
            response = client.models.generate_content(
                model=model,
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    temperature=0.7,
                ),
            )
            _last_call_time[0] = time.time()
            return response.text
        except Exception as e:
            _last_call_time[0] = time.time()
            msg = str(e)
            backoff = min(60, 5 * (2 ** attempt))
            print(f"Gemini 호출 오류(시도 {attempt + 1}/{max_retries}): {msg[:200]} → {backoff}초 대기 후 재시도")
            time.sleep(backoff)
    raise RuntimeError("Gemini 호출이 재시도 한도를 초과했습니다. MIN_SECONDS_BETWEEN_CALLS를 늘리거나 잠시 뒤 다시 시도하세요.")

print("Gemini API 연결 완료")

In [ ]:
# 실제 유동인구 CSV 업로드
# 서울열린데이터광장 "서울시 상권분석서비스(추정유동인구-상권 or 길단위인구-상권)" 원본을
# 그대로 올려도 되고(전체 상권이 다 들어있는 큰 파일), 홍대입구역만 미리 필터링한
# 1행짜리 파일을 올려도 됩니다 — 아래 코드가 알아서 홍대입구역 행을 찾습니다.
uploaded = files.upload()
filename = next(iter(uploaded))


def read_csv_any_encoding(path):
    # 서울열린데이터광장 CSV는 UTF-8일 때도 있고 CP949(EUC-KR)일 때도 있어서
    # 순서대로 시도합니다.
    last_err = None
    for enc in ["utf-8-sig", "utf-8", "cp949", "euc-kr"]:
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError as e:
            last_err = e
            continue
    raise last_err


actual_raw = read_csv_any_encoding(filename)

print("\n업로드된 파일:", filename)
display(actual_raw)

print("\n===== 실제 컬럼명 =====")
for c in actual_raw.columns:
    print(c)

In [ ]:
# 실제값으로 쓸 행을 고릅니다.
# 1) 전체 원본(여러 상권이 섞인 큰 파일)이 업로드된 경우: 상권_코드로 홍대입구역(홍대)만
#    걸러내고, 그 중 지정한 분기(기본 2026 1분기=20261)를 사용합니다.
# 2) 이미 홍대입구역 1행만 필터링된 파일이 업로드된 경우: 그 행을 그대로 사용합니다.
TARGET_SANGKWON_CODE = 3120103   # 홍대입구역(홍대), 발달상권
TARGET_QUARTER = 20261           # 2026년 1분기. 다른 분기를 쓰고 싶으면 이 값을 바꾸세요.

candidates = actual_raw.copy()

if "상권_코드" in candidates.columns:
    by_code = candidates[candidates["상권_코드"] == TARGET_SANGKWON_CODE]
    if not by_code.empty:
        candidates = by_code
    elif "상권_코드_명" in candidates.columns:
        by_name = candidates[candidates["상권_코드_명"].astype(str).str.contains("홍대입구역")]
        if not by_name.empty:
            candidates = by_name

if "기준_년분기_코드" in candidates.columns and (candidates["기준_년분기_코드"] == TARGET_QUARTER).any():
    actual_row = candidates[candidates["기준_년분기_코드"] == TARGET_QUARTER].iloc[0]
else:
    # 지정한 분기가 없으면 '총_유동인구_수'가 가장 큰(=원본) 행을 사용
    actual_row = candidates.loc[candidates["총_유동인구_수"].idxmax()]

print("실제값으로 사용하는 행:")
print(actual_row[["기준_년분기_코드", "상권_코드", "상권_코드_명", "총_유동인구_수"]]
      if "상권_코드_명" in actual_row.index else actual_row[["총_유동인구_수"]])

# 성별
actual_sex_count = {
    "남자": float(actual_row["남성_유동인구_수"]),
    "여자": float(actual_row["여성_유동인구_수"])
}

# 연령
actual_age_count = {
    "10대": float(actual_row["연령대_10_유동인구_수"]),
    "20대": float(actual_row["연령대_20_유동인구_수"]),
    "30대": float(actual_row["연령대_30_유동인구_수"]),
    "40대": float(actual_row["연령대_40_유동인구_수"]),
    "50대": float(actual_row["연령대_50_유동인구_수"]),
    "60대 이상": float(actual_row["연령대_60_이상_유동인구_수"])
}

# 시간대
actual_time_count = {
    "00-06": float(actual_row["시간대_00_06_유동인구_수"]),
    "06-11": float(actual_row["시간대_06_11_유동인구_수"]),
    "11-14": float(actual_row["시간대_11_14_유동인구_수"]),
    "14-17": float(actual_row["시간대_14_17_유동인구_수"]),
    "17-21": float(actual_row["시간대_17_21_유동인구_수"]),
    "21-24": float(actual_row["시간대_21_24_유동인구_수"])
}

# 요일
actual_day_count = {
    "월": float(actual_row["월요일_유동인구_수"]),
    "화": float(actual_row["화요일_유동인구_수"]),
    "수": float(actual_row["수요일_유동인구_수"]),
    "목": float(actual_row["목요일_유동인구_수"]),
    "금": float(actual_row["금요일_유동인구_수"]),
    "토": float(actual_row["토요일_유동인구_수"]),
    "일": float(actual_row["일요일_유동인구_수"])
}

print("성별:", actual_sex_count)
print("연령:", actual_age_count)
print("시간대:", actual_time_count)
print("요일:", actual_day_count)

In [ ]:
def to_pct(d):
    total = sum(d.values())
    return {k: v / total * 100 for k, v in d.items()}

actual_sex = to_pct(actual_sex_count)
actual_time = to_pct(actual_time_count)
actual_day = to_pct(actual_day_count)

# 연령: 10대 포함 원본 비율 + 10대 제외 후 재정규화 (둘 다 비교에 씀)
actual_age_full = to_pct(actual_age_count)

actual_age_no10_count = {k: v for k, v in actual_age_count.items() if k != "10대"}
actual_age_no10 = to_pct(actual_age_no10_count)

print("===== 실제 분포 =====")
print("\n[성별]")
print(actual_sex)
print("\n[연령 - 원본(10대 포함)]")
print(actual_age_full)
print("\n[연령 - 10대 제외 후 정규화]")
print(actual_age_no10)
print("\n[시간대]")
print(actual_time)
print("\n[요일]")
print(actual_day)

In [ ]:
# =========================
# Nemotron Persona 데이터
# =========================

dataset = load_dataset(
    "nvidia/Nemotron-Personas-Korea",
    split="train"
).shuffle(seed=SEED)

print("Nemotron Persona 수:", len(dataset))

# =========================
# 1단계 프롬프트 결과 (홍대) — 사용자가 앞서 만든 1단계 JSON에서
# 한줄요약 + 대표업종_운영시간 + 방문목적만 추려서 2단계 입력으로 사용
# =========================

HONGDAE_INFO = {
    "한줄요약":
        "대학가를 배경으로 카페·맛집·클럽·공연장·소품숍이 밀집해, "
        "평일은 낮부터 저녁까지, 주말은 낮부터 심야·새벽까지 이어지는 "
        "젊은 문화·유흥·쇼핑 거리",

    "대표업종_운영시간": [
        {"업종": "카페", "운영시간": "평일 10시~22시 / 주말 10시~23시 (일부 심야 카페 24시)"},
        {"업종": "음식점·주점", "운영시간": "평일 11시~익일 01시 / 주말 11시~익일 03시"},
        {"업종": "클럽·라이브공연장", "운영시간": "평일 21시~익일 04시 / 주말(금·토) 22시~익일 05시, 주말에 특히 활발"},
        {"업종": "영화관", "운영시간": "10시~24시 (주말 심야 상영 확대)"},
        {"업종": "편의점", "운영시간": "24시간"},
        {"업종": "마트·슈퍼", "운영시간": "10시~23시"},
        {"업종": "호텔·게스트하우스", "운영시간": "24시간 (체크인 오후)"},
        {"업종": "패션·소품 편집숍", "운영시간": "평일 11시~22시 / 주말 11시~22시 (주말 방문객 증가)"}
    ],

    "방문목적": [
        "친구·연인 모임", "카페·맛집 탐방", "야간 유흥·클럽",
        "공연·라이브 관람", "쇼핑·거리 구경", "데이트"
    ]
}

print("\n===== 홍대 상권 정보 (2단계 입력) =====")
print(json.dumps(HONGDAE_INFO, ensure_ascii=False, indent=2))

In [ ]:
AGE_ORDER = ["20대", "30대", "40대", "50대", "60대 이상"]
AGE_ORDER_FULL = ["10대", "20대", "30대", "40대", "50대", "60대 이상"]
SEX_ORDER = ["남자", "여자"]
TIME_ORDER = ["00-06", "06-11", "11-14", "14-17", "17-21", "21-24"]
DAY_ORDER = ["월", "화", "수", "목", "금", "토", "일"]


def age_group(age):
    try:
        num = int(re.findall(r"\d+", str(age))[0])
        if num < 20:
            return "10대"
        elif num < 30:
            return "20대"
        elif num < 40:
            return "30대"
        elif num < 50:
            return "40대"
        elif num < 60:
            return "50대"
        else:
            return "60대 이상"
    except Exception:
        return None


def normalize_sex(x):
    x = str(x).lower()
    if x in ["male", "m", "남", "남자", "남성"]:
        return "남자"
    if x in ["female", "f", "여", "여자", "여성"]:
        return "여자"
    return str(x)


def get_persona_text(p):
    fields = [
        "persona",
        "travel_persona",
        "culinary_persona",
        "hobbies_and_interests",
        "professional_persona"
    ]
    text = []
    for field in fields:
        value = p.get(field)
        if value:
            text.append(f"{field}: {value}")
    return "\n".join(text)[:5000]


def parse_json(text):
    text = text.strip()
    text = re.sub(r"```json|```", "", text).strip()
    start = text.find("[")
    end = text.rfind("]")
    if start != -1 and end != -1:
        text = text[start:end + 1]
    return json.loads(text)


def judge_stage2_batch(batch):
    people = []
    for i, p in enumerate(batch):
        people.append({
            "index": i,
            "나이": p.get("age"),
            "성별": p.get("sex"),
            "직업": p.get("occupation"),
            "취미·성향": get_persona_text(p)
        })

    prompt = f"""
[상권 정보]
{json.dumps(HONGDAE_INFO, ensure_ascii=False)}

[가상 인물]
{json.dumps(people, ensure_ascii=False)}

위 '상권 정보'와 '인물 특성'을 함께 고려해,
각 인물이 향후 30일 내 이 상권을 방문할지 판단하세요.

방문한다면 주로 몇 시경인지와 어느 요일에 방문할지도 판단하세요.

※ 상권 정보는 '이곳이 어떤 장소인지'에 대한 설명일 뿐입니다.
인물의 성향에 비추어 방문 여부와 시간대를 스스로 추론하세요.

실제 방문객의 연령·성별·시간대·요일별 분포나
유동인구 통계를 맞추려고 하지 마세요.

각 인물은 서로 독립적으로 판단하세요.

[출력 — JSON 배열만]

[
  {{
    "index": 0,
    "방문": "예 또는 아니오",
    "주요_시간대": "00-06|06-11|11-14|14-17|17-21|21-24 또는 null",
    "주요_요일": "월|화|수|목|금|토|일 또는 null",
    "이유": "간단한 판단 이유"
  }}
]
"""

    text = paced_generate(prompt)

    return parse_json(text)


print("함수 준비 완료")

In [ ]:
visitors = []
all_decisions = []
dataset_index = 0

print("2단계 프롬프트 실험 시작 (홍대)")

while len(visitors) < TARGET_VISITORS:
    batch = [
        dataset[i]
        for i in range(dataset_index, min(dataset_index + BATCH_SIZE, len(dataset)))
    ]

    try:
        results = judge_stage2_batch(batch)
    except Exception as e:
        print("API 오류 → 같은 batch 재시도:", e)
        continue

    for result in results:
        idx = int(result["index"])
        if idx >= len(batch):
            continue

        person = batch[idx]

        visit = str(result.get("방문", "")).strip() in [
            "예", "yes", "Yes", "YES", "true", "True"
        ]

        record = {
            "uuid": person.get("uuid"),
            "age": person.get("age"),
            "age_group": age_group(person.get("age")),
            "sex": normalize_sex(person.get("sex")),
            "occupation": person.get("occupation"),
            "visit": visit,
            "time_slot": result.get("주요_시간대") if visit else None,
            "weekday": result.get("주요_요일") if visit else None,
            "reason": result.get("이유", "")
        }

        all_decisions.append(record)

        if visit:
            visitors.append(record)
            print(
                f"방문 {len(visitors):3d}/{TARGET_VISITORS} | "
                f"{record['age_group']} | {record['sex']} | "
                f"{record['time_slot']} | {record['weekday']}"
            )

        if len(visitors) >= TARGET_VISITORS:
            break

    dataset_index += BATCH_SIZE

stage2_visitors = pd.DataFrame(visitors[:TARGET_VISITORS])
stage2_all = pd.DataFrame(all_decisions)

print("\n===== 2단계 실험 종료 (홍대) =====")
print("평가 Persona:", len(stage2_all))
print("방문자:", len(stage2_visitors))
print("수락률:", round(len(stage2_visitors) / len(stage2_all) * 100, 2), "%")

display(stage2_visitors.head())

stage2_visitors.to_csv("hongdae_stage2_visitors_500.csv", index=False, encoding="utf-8-sig")
stage2_all.to_csv("hongdae_stage2_all_decisions.csv", index=False, encoding="utf-8-sig")

## 베이스라인: 2단계(방문판단) 없이 페르소나 풀에서 무작위 추출

숙제에서 요청한 "2단계 안 하고 10대만 정규화했을 때"와 비교하기 위한 베이스라인입니다.
페르소나 풀(100만 명)에서 무작위로 `BASELINE_N`명을 뽑아, 그 자체의 나이/성별 분포를
10대 제외 후 정규화해서 봅니다. LLM 방문판단을 전혀 거치지 않은, "그냥 일반 인구 표본"에
가깝습니다. 시간대·요일은 이 베이스라인에 값이 없으므로 계산하지 않습니다.

In [ ]:
random.seed(SEED)
baseline_indices = random.sample(range(len(dataset)), BASELINE_N)

baseline_records = []
for i in baseline_indices:
    p = dataset[i]
    baseline_records.append({
        "uuid": p.get("uuid"),
        "age": p.get("age"),
        "age_group": age_group(p.get("age")),
        "sex": normalize_sex(p.get("sex")),
        "occupation": p.get("occupation")
    })

baseline_df = pd.DataFrame(baseline_records)
print("베이스라인 표본 수:", len(baseline_df))
display(baseline_df.head())

baseline_df.to_csv("hongdae_baseline_sample.csv", index=False, encoding="utf-8-sig")

In [ ]:
def distribution(df, column, order, exclude_10=False):
    temp = df.copy()
    if exclude_10:
        temp = temp[temp["age_group"] != "10대"]
    counts = temp[column].value_counts().reindex(order, fill_value=0)
    return (counts / counts.sum() * 100).to_dict()


def make_compare(actual, simulation, title):
    rows = []
    for category in actual:
        actual_value = actual[category]
        sim_value = simulation.get(category, 0)
        rows.append({
            "구분": category,
            "실제(%)": round(actual_value, 2),
            "시뮬레이션(%)": round(sim_value, 2),
            "오차(%p)": round(abs(sim_value - actual_value), 2)
        })

    result = pd.DataFrame(rows)
    mae = result["오차(%p)"].mean()

    print("\n" + "=" * 50)
    print(title)
    print("=" * 50)
    display(result)
    print("MAE:", round(mae, 2), "%p")

    return result, mae

In [ ]:
# ===== 2단계 시뮬레이션 결과 vs 실제 =====

stage2_age = distribution(stage2_visitors, "age_group", AGE_ORDER, exclude_10=True)
stage2_sex = distribution(stage2_visitors, "sex", SEX_ORDER)
stage2_time = distribution(stage2_visitors, "time_slot", TIME_ORDER)
stage2_day = distribution(stage2_visitors, "weekday", DAY_ORDER)

age_result, age_mae = make_compare(actual_age_no10, stage2_age, "[2단계] 연령대 비교 - 10대 제외 정규화")
sex_result, sex_mae = make_compare(actual_sex, stage2_sex, "[2단계] 성별 비교")
time_result, time_mae = make_compare(actual_time, stage2_time, "[2단계] 시간대 비교")
day_result, day_mae = make_compare(actual_day, stage2_day, "[2단계] 요일 비교")

summary_stage2 = pd.DataFrame([
    {"항목": "연령대(10대 제외)", "MAE(%p)": round(age_mae, 2)},
    {"항목": "성별", "MAE(%p)": round(sex_mae, 2)},
    {"항목": "시간대", "MAE(%p)": round(time_mae, 2)},
    {"항목": "요일", "MAE(%p)": round(day_mae, 2)}
])

print("\n===== 2단계 최종 MAE (홍대) =====")
display(summary_stage2)

In [ ]:
# ===== 베이스라인(2단계 없음) vs 실제 — 연령대·성별만 =====

baseline_age = distribution(baseline_df, "age_group", AGE_ORDER, exclude_10=True)
baseline_sex = distribution(baseline_df, "sex", SEX_ORDER)

baseline_age_result, baseline_age_mae = make_compare(
    actual_age_no10, baseline_age, "[베이스라인 · 2단계 없음] 연령대 비교 - 10대 제외 정규화"
)
baseline_sex_result, baseline_sex_mae = make_compare(
    actual_sex, baseline_sex, "[베이스라인 · 2단계 없음] 성별 비교"
)

# 10대 포함 원본 vs 10대 제외 정규화 — 정규화 자체의 효과 (2단계 결과 기준)
stage2_age_full = distribution(stage2_visitors, "age_group", AGE_ORDER_FULL, exclude_10=False)
_, age_mae_full = make_compare(actual_age_full, stage2_age_full, "[2단계] 연령대 비교 - 원본(10대 포함)")

compare_table = pd.DataFrame([
    {"비교": "연령대 MAE - 원본(10대 포함)", "값(%p)": round(age_mae_full, 2)},
    {"비교": "연령대 MAE - 10대 제외 정규화 (2단계)", "값(%p)": round(age_mae, 2)},
    {"비교": "연령대 MAE - 10대 제외 정규화 (베이스라인, 2단계 없음)", "값(%p)": round(baseline_age_mae, 2)},
    {"비교": "성별 MAE - 2단계", "값(%p)": round(sex_mae, 2)},
    {"비교": "성별 MAE - 베이스라인(2단계 없음)", "값(%p)": round(baseline_sex_mae, 2)},
])

print("\n===== 종합: 10대 정규화 효과 + 2단계 유무 효과 (홍대) =====")
display(compare_table)

print(f"\n10대 제외 정규화가 연령 MAE를 바꾼 정도(2단계 기준): {age_mae_full:.2f} -> {age_mae:.2f} ({age_mae - age_mae_full:+.2f}%p)")
print(f"2단계 도입이 연령 MAE를 바꾼 정도(둘 다 10대 제외 정규화): 베이스라인 {baseline_age_mae:.2f} -> 2단계 {age_mae:.2f} ({age_mae - baseline_age_mae:+.2f}%p)")
print(f"2단계 도입이 성별 MAE를 바꾼 정도: 베이스라인 {baseline_sex_mae:.2f} -> 2단계 {sex_mae:.2f} ({sex_mae - baseline_sex_mae:+.2f}%p)")

In [ ]:
age_result["항목"] = "연령대"
sex_result["항목"] = "성별"
time_result["항목"] = "시간대"
day_result["항목"] = "요일"

final_result = pd.concat([age_result, sex_result, time_result, day_result], ignore_index=True)
final_result = final_result[["항목", "구분", "실제(%)", "시뮬레이션(%)", "오차(%p)"]]
final_result.to_csv("hongdae_stage2_final_comparison_500.csv", index=False, encoding="utf-8-sig")

compare_table.to_csv("hongdae_normalization_and_stage2_effect.csv", index=False, encoding="utf-8-sig")

print("저장 완료: hongdae_stage2_final_comparison_500.csv, hongdae_normalization_and_stage2_effect.csv")
display(final_result)